# v40 — быстрые G1/G2 и BN

Run All на текущем Mac. Без optimizer updates, без изменения MVP, bbox и порога.
17 вариантов fusion на готовых признаках: выбор на calibration, затем только контроль и победитель на validation.
BN: R1 и N1, до/после одного прохода **только primary train**. Полный набор экспериментов GPU — в v41.
Время не ограничено; быстрее обучения, но BN требует чтения изображений и encoder-forward.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['XFORMERS_DISABLED'] = '1'
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/research_gpu.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри клона Car-classification-MSK / fine-tuning')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
INPUTS = ROOT / 'research_transfer/v41_inputs'
print('Python:', sys.executable)
print('Repo:', ROOT)
print('Inputs:', INPUTS)
if not (INPUTS / 'inputs.json').is_file():
    raise FileNotFoundError('Распакуй v41_inputs.zip в research_transfer/v41_inputs. См. README варианта.')
RUN_NAME = 'quick_v1'
DEVICE = 'mps'  # 'cpu' только если MPS недоступен; автоматической подмены нет.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_parallel_research.py'], cwd=ROOT, env=os.environ.copy(), check=True)

In [ ]:
from training.research_quick import run
summary = run(INPUTS, RUN_NAME, DEVICE)


In [ ]:
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Отдать на анализ:', ROOT / f'OSNet-AIN-x1.0/variant_40_quick_diagnostics/{RUN_NAME}_analysis.zip')
